In [ ]:
"""
File Name: pol_entropy_image_saocom.ipynb

Authors:
    Juan M. Lopez-Sanchez
    University of Alicante

Date: July 2025

Description:
------------
This script computes the entropy of a dual-pol (VV-VH) SAOCOM image and saves
it as a GeoTIFF file.

The original image must have been previously processed and should be available
in POL format (either as COV or GCOV).

Dependencies:
-------------
- numpy
- os
- pathlib

- rosel.io
- rosel.computation
- rosel.shiroma


Usage:
------
To use this script, define the following parameters:

    - common_folder: Common path to all products, e.g. '/media/s3-ard-rose-l/PHASE_1/DEMONSTRATOR'
    - site: 'Naples', 'Salamanca', or 'BlackForest'
    - orbit: 'ASC' or 'DSC'
    - beam: 'S3' or 'S4'
    - multilook_option: 'SHIROMA' (without multi-look) or 'MULTI-LOOK'
    - multilook_size: e.g. '9x5', '11x7', 15x9', etc. Not used if multilook_option is 'SHIROMA'
    - product: 'COV' or 'GCOV'
    - acquisition_date: time stamp of the original product, from with the date is used, e.g. '2021-02-28T17:09:09.724349' # Available for DSC/S4 Naples
"""

Package dependencies

In [1]:
import numpy as np
import os
import pathlib

import rosel.io
import rosel.computation
import rosel.shiroma

Auxiliary function for computing the dual-pol entropy from the covariance matrix

In [2]:
def dualpol_entropy(C11, C22, C12):

    # PROCESSING

    Nrows, Ncols = C11.shape

    # Convert to numpy arrays for using 2-D masks

    c11 = np.array(np.abs(C11)).astype(np.float32)
    c22 = np.array(np.abs(C22)).astype(np.float32)
    c12 = np.array(C12).astype(np.complex64)

    # Initialization

    H = np.zeros((Nrows, Ncols))

    C2 = np.zeros((2, 2), dtype=np.complex64)

    for ir in range(Nrows):
        for ic in range(Ncols):

            if c11[ir, ic] >= 1e-10:

                C2[0, 0] = c11[ir, ic]
                C2[0, 1] = c12[ir, ic]
                C2[1, 0] = np.conj(C2[0, 1])
                C2[1, 1] = c22[ir, ic]

                eigenvalues, eigenvectors = np.linalg.eig(C2)

                span = eigenvalues[0] + eigenvalues[1]
                p1 = eigenvalues[0] / span
                p2 = eigenvalues[1] / span

                entropy = -p1 * np.log2(p1) - p2 * np.log2(p2)

                H[ir, ic] = entropy

    return H

User-defined input parameters 

In [3]:
########################
# PARAMETERS
########################


common_folder = "/media/s3-ard-rose-l/PHASE_1/DEMONSTRATOR"

site = "Naples"  # Naples, Salamanca, BlackForest
orbit = "DSC"  # ASC, DSC
beam = "S4"  # S3, S4

multilook_option = "MULTI-LOOK"  # SHIROMA or MULTI-LOOK
multilook_size = "11x7"  # e.g. '9x5', '11x7', 15x9'

product = "GCOV"  # 'COV' or 'GCOV'

# Date selection
# Example
acquisition_date = "2021-02-28T17:09:09.724349"  # DSC/S4 Naples

Definition of input folder with the POL product to be used (by means of the user-defined parameters)

In [4]:
# Date format (as subfolder of POL products)

date_txt = acquisition_date[:10]  # only the string with the date

# Form path to input POL product

polsar_folder = os.path.join(common_folder, "POL", site, orbit, beam, multilook_option)

if multilook_option == "MULTI-LOOK":
    polsar_folder = os.path.join(polsar_folder, multilook_size)

input_folder = os.path.join(polsar_folder, date_txt)

Output folder to save the entropy image

In [5]:
output_folder = os.path.join(input_folder, "figures")
pathlib.Path(output_folder).mkdir(parents=True, exist_ok=True)

PROCESSING

In [6]:
# Read the 3 layers of a POL product

C11 = rosel.io.read(
    os.path.join(input_folder, product, "C11"), consolidate_metadata=False
).v
C22 = rosel.io.read(
    os.path.join(input_folder, product, "C22"), consolidate_metadata=False
).v
C12 = rosel.io.read(
    os.path.join(input_folder, product, "C12"), consolidate_metadata=False
).v

In [7]:
# Computation of entropy of a dual-pol POL product

H2 = dualpol_entropy(C11, C22, C12)

In [8]:
# Reformat as a (geocoded) dataset

H2_dataset = abs(C22)
H2_dataset[:] = H2[:]

# Output file name

file_tiff = os.path.join(output_folder, "Entropy.tiff")

# Save output file as GEOTIFF (valid for an input product in GCOV format)

H2_dataset.rio.to_raster(file_tiff, driver="COG")